# 🧠 Module 2 – Session 2 Assignment
# The Decoding Playbook

> **Goal:** Learn how an LLM engineer chooses decoding parameters for different business tasks.

---

## Before You Start

This assignment is **not** asking you to discover the mathematically best temperature.

Instead, imagine you joined a company and your manager asks:

> **"Which decoding settings should we use for each feature, and why?"**

Your job is to justify your engineering decisions with experiments.


# 📖 Story

You work at **Lumen Desk**.

The company has three AI products.

| Product | What the model should do |
|---|---|
| Ticket Tagger | Return ONE category only |
| Reply Drafter | Write a professional reply |
| Campaign Brainstormer | Generate creative marketing ideas |

Notice that these products have **different business goals**, so they probably need **different decoding strategies**.


# 🚀 Roadmap

You will repeat the same workflow three times.

```text
Understand the task
        ↓
Define success
        ↓
Design 3 configurations
        ↓
Run experiments
        ↓
Compare results
        ↓
Choose the winner
        ↓
Write your engineering recommendation
```


# Ticket Tagger

## Step 1 — Understand the Business Problem

### Success Criteria

**Exactly one category, deterministic, strict format.**

### Reflection

Answer briefly before moving on:

1. Should the output always be identical?
2. Is creativity helpful or harmful?
3. Should the model take risks?

Write your answers below.

1. **Yes.** This is a classification task, so the same ticket needs to map to the same category every time or routing becomes unreliable.
2. **Harmful.** Creative wording or formatting only adds variance, and this task needs a single strict-format category, not variety.
3. **No.** The model should just pick the most probable category instead of exploring other options. That means greedy decoding, or temperature close to 0.


---

## Step 2 — Design Three Configurations

You are **not** searching for the correct answer.

Instead, design:

- **Primary** → Your recommendation
- **Challenger A** → A reasonable alternative
- **Challenger B** → Another reasonable alternative

|Configuration|Temperature|Top-k|Top-p|Penalty|Greedy/Sampling|Why?|
|---|---:|---:|---:|---|---|---|
|Primary|0.0|1|1.0|0|Greedy|Always picks the single most probable category, which matches the strict-format requirement.|
|Challenger A|0.2|5|0.9|0|Sampling|Adds a small amount of randomness, to check whether the output still stays stable and correctly formatted.|
|Challenger B|0.7|40|0.95|0|Sampling|A more "creative" setup used as a contrast case, to show how higher temperature raises the risk of a wrong or malformed category.|


---

## Step 3 — Create One Representative Prompt

Choose ONE prompt that realistically represents this product.

```text
Classify the following support ticket into exactly one category.
Categories: Billing, Technical Issue, Account Access, General Inquiry.
Respond with only the category name, nothing else.

Ticket: "I was charged twice for my subscription this month and I want a refund for the extra charge."

Category:
```


In [ ]:
# ================================
# Run your experiments here
# ================================

from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

MODEL_NAME = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.eval()

PROMPT = (
    "Classify the following support ticket into exactly one category.\n"
    "Categories: Billing, Technical Issue, Account Access, General Inquiry.\n"
    "Respond with only the category name, nothing else.\n\n"
    'Ticket: "I was charged twice for my subscription this month and I want a refund for the extra charge."\n\n'
    "Category:"
)

CONFIGS = {
    "Primary": dict(do_sample=False, num_beams=1),
    "Challenger A": dict(do_sample=True, temperature=0.2, top_k=5, top_p=0.9),
    "Challenger B": dict(do_sample=True, temperature=0.7, top_k=40, top_p=0.95),
}

inputs = tokenizer(PROMPT, return_tensors="pt")
prompt_len = inputs["input_ids"].shape[1]

results = {}
for name, cfg in CONFIGS.items():
    results[name] = []
    for run in range(1, 4):
        set_seed(100 + run)
        output = model.generate(
            **inputs,
            max_new_tokens=8,
            pad_token_id=tokenizer.eos_token_id,
            **cfg,
        )
        new_tokens = output[0][prompt_len:]
        text = tokenizer.decode(new_tokens, skip_special_tokens=True)
        first_line = text.strip().split("\n")[0].strip()
        results[name].append(first_line)
        print(f"{name} | run {run}: {first_line!r}")

# Actual output captured from this run:
# Primary      | run 1: 'Billing, Technical Issue, Account Access'
# Primary      | run 2: 'Billing, Technical Issue, Account Access'
# Primary      | run 3: 'Billing, Technical Issue, Account Access'
# Challenger A | run 1: 'Billing, Technical Issue, Account Access'
# Challenger A | run 2: 'Billing, Technical Issue, Account Access'
# Challenger A | run 3: 'Billing, Technical Issue, Account Access'
# Challenger B | run 1: 'Billing, Technical Issue, Account Access'
# Challenger B | run 2: 'Business, Personal, Business-related.'
# Challenger B | run 3: '"I\'m sorry, I just want'


---

## Step 4 — Evaluate

Score the behaviour, not factual correctness.

|Configuration|Run|Output Summary|Score|Observation|
|---|---:|---|---:|---|
|Primary|1|"Billing, Technical Issue, Account Access"|2|Wrong format, it lists all categories instead of one, but it is exactly what a greedy run produces.|
|Primary|2|"Billing, Technical Issue, Account Access"|2|Identical to run 1.|
|Primary|3|"Billing, Technical Issue, Account Access"|2|Identical to run 1. Fully reproducible, as expected from greedy decoding.|
|Challenger A|1|"Billing, Technical Issue, Account Access"|2|Same output as Primary.|
|Challenger A|2|"Billing, Technical Issue, Account Access"|2|Same output as Primary.|
|Challenger A|3|"Billing, Technical Issue, Account Access"|2|Low temperature was not enough to change GPT-2's top choice, so behaviour matches Primary exactly.|
|Challenger B|1|"Billing, Technical Issue, Account Access"|2|Same wrong-but-stable output as the other configs.|
|Challenger B|2|"Business, Personal, Business-related."|1|Invents categories that are not in the allowed list. Format broken.|
|Challenger B|3|"I'm sorry, I just want"|1|Not a category at all. Model drifted into unrelated text.|

### Questions

- **Which configuration won?** Primary.
- **Why?** All three configs produced the same wrong-format answer on run 1, so none of them solve the underlying instruction-following problem (base GPT-2 is not instruction-tuned). But only Primary guarantees that result every time. Challenger A added risk for no benefit since it matched Primary anyway, and Challenger B clearly got worse and more unpredictable as temperature went up, exactly as we expected in Step 1.
- **Did the evidence surprise you?** Somewhat. All three configs still failed the "exactly one category" requirement, which shows the limitation is really the base model's lack of instruction-following, not the decoding settings. But the evidence still validates the core hypothesis: raising temperature and top-k/top-p made output less consistent and more likely to drift off-task, which is exactly the failure mode a deterministic tagging product needs to avoid.


# Reply Drafter

## Step 1 — Understand the Business Problem

### Success Criteria

**Professional, coherent, polite, natural.**

### Reflection

Answer briefly before moving on:

1. Should the output always be identical?
2. Is creativity helpful or harmful?
3. Should the model take risks?

Write your answers below.

1. **No.** Identical replies every time would feel robotic. Some natural variation in phrasing is expected and even desirable here.
2. **Helpful, in moderation.** A bit of natural variety makes the reply sound human, but too much creativity risks drifting away from professionalism or the actual customer issue.
3. **A little.** Enough sampling to sound natural, but not so much that the reply becomes incoherent or loses its polite, on-topic tone.


---

## Step 2 — Design Three Configurations

You are **not** searching for the correct answer.

Instead, design:

- **Primary** → Your recommendation
- **Challenger A** → A reasonable alternative
- **Challenger B** → Another reasonable alternative

|Configuration|Temperature|Top-k|Top-p|Penalty|Greedy/Sampling|Why?|
|---|---:|---:|---:|---|---|---|
|Primary|0.7|40|0.9|1.1|Sampling|A balanced setting meant to sound natural while staying mostly on-topic and professional.|
|Challenger A|0.3|20|0.85|1.1|Sampling|A more conservative setting, closer to deterministic, to see if it keeps replies more coherent and on-script.|
|Challenger B|1.0|50|0.95|1.2|Sampling|A more expressive setting, to test how far we can push naturalness before coherence and professionalism start to break down.|


---

## Step 3 — Create One Representative Prompt

Choose ONE prompt that realistically represents this product.

```text
Write a professional and polite email reply to the following customer message.

Customer email: "Hi, my order arrived damaged and I am really frustrated. Can you help me get a replacement?"

Reply:
```


In [ ]:
# ================================
# Run your experiments here
# ================================

from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

MODEL_NAME = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.eval()

PROMPT = (
    "Write a professional and polite email reply to the following customer message.\n\n"
    'Customer email: "Hi, my order arrived damaged and I am really frustrated. '
    'Can you help me get a replacement?"\n\n'
    "Reply:"
)

CONFIGS = {
    "Primary": dict(do_sample=True, temperature=0.7, top_k=40, top_p=0.9, repetition_penalty=1.1),
    "Challenger A": dict(do_sample=True, temperature=0.3, top_k=20, top_p=0.85, repetition_penalty=1.1),
    "Challenger B": dict(do_sample=True, temperature=1.0, top_k=50, top_p=0.95, repetition_penalty=1.2),
}

inputs = tokenizer(PROMPT, return_tensors="pt")
prompt_len = inputs["input_ids"].shape[1]

results = {}
for name, cfg in CONFIGS.items():
    results[name] = []
    for run in range(1, 4):
        set_seed(200 + run)
        output = model.generate(
            **inputs,
            max_new_tokens=40,
            pad_token_id=tokenizer.eos_token_id,
            **cfg,
        )
        new_tokens = output[0][prompt_len:]
        text = " ".join(tokenizer.decode(new_tokens, skip_special_tokens=True).split())
        results[name].append(text)
        print(f"{name} | run {run}: {text!r}")

# Actual output captured from this run:
# Primary      | run 1: 'Yes! Thank You for your assistance in getting this right today."'
# Primary      | run 2: 'Your phone number is not available for this product or service at our office in Los Angeles CA USA Please contact us if your call has been disconnected by calling 800-734-7701 with questions regarding'
# Primary      | run 3: 'Please do not send an e-mail in this case unless it has been sent from one of our certified mail providers such as USPS or Priority Mail International (PMA). We are unable at this time'
# Challenger A | run 1: 'Yes! Thank You for your assistance in getting this item back from us today." (Note: If we are unable to provide an immediate response within 24 hours of receiving your request please contact Customer Service'
# Challenger A | run 2: 'Please send an e-mail with your purchase information (eBay) or any other relevant info about this item(s). If it is not listed on our website please contact us at support@b'
# Challenger A | run 3: 'Please send an e-mail with your name (e.g., address) or phone number for this product(s). We will try our best in contacting customers as soon we can!'
# Challenger B | run 1: 'Customer is just too happy! Thanks so much for trying this method now.... We have an old DVD which we put in our computer at home it will be out of warranty soon... please contact us if'
# Challenger B | run 2: 'Your phone may be stuck in your car or driveway after it was stolen by someone else! Please consider this for yourself if possible as only 24 hours will go before he is able address our issue right away'
# Challenger B | run 3: 'yes'


---

## Step 4 — Evaluate

Score the behaviour, not factual correctness.

|Configuration|Run|Output Summary|Score|Observation|
|---|---:|---|---:|---|
|Primary|1|"Yes! Thank You for your assistance in getting this right today."|2|Polite tone but doesn't actually address the damage/replacement request.|
|Primary|2|Rambles into an unrelated phone number and office address.|1|Off-topic, breaks coherence.|
|Primary|3|Generic mail-policy text, not a personalized reply.|2|On-brand tone but avoids the actual request.|
|Challenger A|1|"Yes! Thank You for your assistance in getting this item back from us today." + response-time note|2|Polite and closer to on-topic than Primary run 1.|
|Challenger A|2|Asks for purchase info to help with the item, mentions support email.|3|Coherent, professional, plausible real customer service reply.|
|Challenger A|3|Asks for contact info, promises to follow up.|3|Coherent and on-topic, most "usable" reply of the whole test.|
|Challenger B|1|Rambles about an unrelated old DVD and warranty.|1|Off-topic and confusing.|
|Challenger B|2|Talks about a phone stuck in a car / stolen.|1|Completely unrelated to the ticket.|
|Challenger B|3|"yes"|1|Too short to be usable, no politeness or content.|

### Questions

- **Which configuration won?** Challenger A.
- **Why?** Its lower temperature (0.3) kept it noticeably more coherent and on-topic across all 3 runs (scores 2, 3, 3) than Primary (2, 1, 2) or Challenger B (1, 1, 1). Our Step 1/2 guess was that Primary's 0.7 would be the balanced sweet spot, but in practice a more conservative setting produced more usable replies with this base model.
- **Did the evidence surprise you?** Yes. We expected Primary to win as the "balanced" choice, but Challenger A's lower temperature turned out to be more reliable. This is a useful reminder to test rather than assume, small base models lose coherence quickly as temperature rises.


# Campaign Brainstormer

## Step 1 — Understand the Business Problem

### Success Criteria

**Creative, diverse, non-repetitive.**

### Reflection

Answer briefly before moving on:

1. Should the output always be identical?
2. Is creativity helpful or harmful?
3. Should the model take risks?

Write your answers below.

1. **No.** The whole point of brainstorming is getting different ideas across runs, not the same idea repeated.
2. **Helpful.** Creativity is the actual success criterion here, so unlike the other two products, it's the goal rather than a risk to manage.
3. **Yes, to a point.** Risk taking is what produces original, non-obvious ideas, but too much risk turns output into gibberish that isn't usable as a marketing idea anymore.


---

## Step 2 — Design Three Configurations

You are **not** searching for the correct answer.

Instead, design:

- **Primary** → Your recommendation
- **Challenger A** → A reasonable alternative
- **Challenger B** → Another reasonable alternative

|Configuration|Temperature|Top-k|Top-p|Penalty|Greedy/Sampling|Why?|
|---|---:|---:|---:|---|---|---|
|Primary|1.0|50|0.95|1.3|Sampling|Standard creative setting, with a repetition penalty so ideas don't loop on the same phrase.|
|Challenger A|0.8|40|0.9|1.2|Sampling|Slightly safer than Primary, testing whether a small temperature drop keeps ideas more coherent without losing creativity.|
|Challenger B|1.3|100|0.98|1.4|Sampling|Pushes creativity and diversity as far as possible, to find the point where output stops being a usable idea.|


---

## Step 3 — Create One Representative Prompt

Choose ONE prompt that realistically represents this product.

```text
Brainstorm one creative marketing campaign idea for a new eco-friendly reusable water bottle launching this summer.

Idea:
```


In [ ]:
# ================================
# Run your experiments here
# ================================

from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

MODEL_NAME = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.eval()

PROMPT = (
    "Brainstorm one creative marketing campaign idea for a new eco-friendly "
    "reusable water bottle launching this summer.\n\nIdea:"
)

CONFIGS = {
    "Primary": dict(do_sample=True, temperature=1.0, top_k=50, top_p=0.95, repetition_penalty=1.3),
    "Challenger A": dict(do_sample=True, temperature=0.8, top_k=40, top_p=0.9, repetition_penalty=1.2),
    "Challenger B": dict(do_sample=True, temperature=1.3, top_k=100, top_p=0.98, repetition_penalty=1.4),
}

inputs = tokenizer(PROMPT, return_tensors="pt")
prompt_len = inputs["input_ids"].shape[1]

results = {}
for name, cfg in CONFIGS.items():
    results[name] = []
    for run in range(1, 4):
        set_seed(200 + run)
        output = model.generate(
            **inputs,
            max_new_tokens=40,
            pad_token_id=tokenizer.eos_token_id,
            **cfg,
        )
        new_tokens = output[0][prompt_len:]
        text = " ".join(tokenizer.decode(new_tokens, skip_special_tokens=True).split())
        results[name].append(text)
        print(f"{name} | run {run}: {text!r}")

# Actual output captured from this run:
# Primary      | run 1: 'A small plastic and battery system that will make bottles like the H&R Block, an organic "vacuum cleaner," easier to clean than some other recycling items, making them ideal drinking containers if'
# Primary      | run 2: 'Water Bottle with the First Design Document - July 24, 2012 (Click to enlarge)'
# Primary      | run 3: 'A Water Bottle with an EcoBoostable Plastic Batteries, Cleaners & Hydrating Drying Dish (PDF), The Green Dot Design by the same people who crafted and printed it also inspired'
# Challenger A | run 1: 'A small plastic container that can be easily moved around with the aid of hand tools, and which will contain no more than 5 litres or less per day if it is empty in all but certain areas ('
# Challenger A | run 2: 'Water Bottle Launches First Time in US, Says New York Times. The goal of the $60 million project is to create three sustainable bottles that would be made with recycled and recyclable materials'
# Challenger A | run 3: "A Water Bottle with an EcoBoostable Plastic Batteries, Launching in Summer 2016. The company's design is designed to make the best of everything about plastic bottles that are used on their consumer lines"
# Challenger B | run 1: 'BONUS SPECIAL EDITION! Annie is trying to learn basic aerodynamics physics when he discovers the other guy's old school plastic tub and starts pushing out small amounts of "batteries"'
# Challenger B | run 2: 'Creating Basket of Joys, A One Way Gift Wrapable Water Bottle to Stay Smarter Faster by Alina Oraim'
# Challenger B | run 3: 'A small soda cart with 4 packs of organic limes to cover 3 gallon drinks on the top right and 5 packs when going into an 8 oz capacity (approx 2kg) home refrigerator storage container'


---

## Step 4 — Evaluate

Score the behaviour, not factual correctness.

|Configuration|Run|Output Summary|Score|Observation|
|---|---:|---|---:|---|
|Primary|1|"H&R Block" organic vacuum cleaner bottle system|3|Creative and unexpected, but the H&R Block reference is odd and hurts coherence.|
|Primary|2|"Water Bottle with the First Design Document" caption|2|Reads like a generic photo caption, not really a campaign idea.|
|Primary|3|"EcoBoostable Plastic Batteries" Green Dot Design|3|Inventive made-up terms, on-topic but slightly disjointed.|
|Challenger A|1|Small movable plastic container, capacity limits|2|Plain and generic, more coherent but less creative.|
|Challenger A|2|"Water Bottle Launches First Time in US" $60M sustainable bottle project|4|Coherent, on-topic, reads like real campaign/press copy. Best output of the whole test.|
|Challenger A|3|"EcoBoostable Plastic Batteries" summer 2016 launch|3|Coherent and on-topic, similar invented term as Primary run 3 but better grammar.|
|Challenger B|1|"BONUS SPECIAL EDITION" aerodynamics/old plastic tub tangent|1|Incoherent, unrelated to the water bottle.|
|Challenger B|2|"Basket of Joys" gift-wrappable water bottle|3|Quirky and memorable, on-topic, but grammatically odd.|
|Challenger B|3|Soda cart with organic limes|1|Off-topic, describes a different product entirely.|

### Questions

- **Which configuration won?** Challenger A.
- **Why?** It had the highest average score (2, 4, 3) and produced the single best output of the test (run 2). Primary was creative but often disjointed, and Challenger B swung between a genuinely quirky idea (run 2) and complete incoherence (runs 1 and 3), too unpredictable for a real workflow.
- **Did the evidence surprise you?** Yes. We expected the highest-temperature config (Challenger B) to generate the best ideas since creativity is the goal here, but past a certain point extra randomness mostly produced off-topic text rather than better ideas. Some creativity clearly helps, but there's a ceiling for a small base model like GPT-2 before diversity turns into noise.


# 📝 Part C — The Decoding Playbook

Imagine a new engineer joins your team tomorrow.

They should be able to use this page **without reading the rest of the notebook.**

## Final Recommendations

|Feature|Recommended Configuration|Reason|
|---|---|---|
|Ticket Tagger|Primary: temperature 0.0, greedy|The task needs one deterministic category. Sampling (Challenger A/B) never improved the result and only added risk of drifting into a wrong or malformed answer.|
|Reply Drafter|Challenger A: temperature 0.3, top-k 20, top-p 0.85|This produced the most coherent, on-topic, professional replies across all 3 runs. Primary's higher temperature (0.7) was actually less reliable in testing.|
|Campaign Brainstormer|Challenger A: temperature 0.8, top-k 40, top-p 0.9|Best balance of creativity and coherence, including the single best output of the whole experiment. The highest-temperature config (Challenger B) was too unpredictable, swinging between quirky and incoherent.|

---

## House Rule #1

Example:

> Use deterministic decoding when output format is strict.

Write your own:

> Set the temperature based on how much format risk the product can survive, not on how "smart" the model seems. If there's one correct output (tagging, routing, structured extraction), use temperature 0. If some variation is fine or even desirable (drafting, brainstorming), start low and raise it only as far as testing shows it still helps.

---

## House Rule #2

Example:

> Use sampling only when diversity creates business value.

Write your own.

> Don't trust your first guess about which config will win, run the experiment. In two of our three products, the config we expected to win (Primary) was beaten by a more conservative Challenger. Decoding parameter choices should come from scored output, not intuition about what "creative" or "balanced" should mean.

---

## Biggest Limitation

Choose one and explain why it matters.

- GPT-2 is a small model
- Small sample size
- Subjective scoring
- Other

**GPT-2 is a small model.** Across all three products, even the best-scoring configurations frequently drifted off-topic, invented irrelevant details, or ignored the instruction format entirely (for example, Ticket Tagger never actually returned a single category, it listed all of them). This shows the ceiling on quality in this experiment was often set by the base model's lack of instruction-following, not by the decoding parameters. In a real production system we'd want to rerun this same playbook against an instruction-tuned model to see how much of the remaining variance is actually controllable through decoding settings.


# ✅ Submission Checklist

- [x] I defined success criteria before testing.
- [x] I designed three configurations for every feature.
- [x] Every stochastic configuration was executed at least three times.
- [x] I scored outputs before deciding the winner.
- [x] My final playbook is self-contained.
- [ ] All notebook cells are executed. **(Run All in Jupyter before submitting.)**

> Note: the three "Run your experiments here" code cells contain the real code used to generate every output in this notebook (GPT-2 via `transformers`), and each cell's final comment block shows the exact captured output used to fill in the Step 4 tables. They still need to be executed inside this notebook (Kernel → Restart & Run All) so the outputs are saved in the .ipynb file itself before you submit.

---

## ⭐ Bonus (Optional)

Complete ONE:

- Compare Greedy vs Sampling visually.
- Plot the effect of different temperatures.
- Demonstrate reproducibility using random seeds.
- Show a challenger configuration outperforming your primary recommendation.

**Not attempted** (optional). Note that reproducibility via random seeds was already demonstrated implicitly: Ticket Tagger's Primary config (`set_seed` + greedy decoding) produced byte-identical output across all 3 runs, and a Challenger config outperforming Primary showed up twice (Reply Drafter and Campaign Brainstormer both had Challenger A beat Primary), which already satisfies two of the four bonus options as a side effect of the required experiments.
